[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/04_%E3%83%81%E3%83%A3%E3%83%8D%E3%83%AB%E5%88%A5%E3%82%AF%E3%83%AA%E3%83%83%E3%82%AF%E7%8E%87%E3%81%AE%E6%AF%94%E8%BC%83.ipynb)

In [ ]:
# === ① セットアップ（最初に実行してください）===
import pandas as pd               # 表データ
import numpy as np                # 数値計算
import matplotlib.pyplot as plt   # グラフ描画
import os
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の文字化け防止
# ローカルに data/ が無ければ公開リポジトリ(raw)から読み込む
RAW = 'https://raw.githubusercontent.com/Carlo-Broschi/statistics-python-for-students/main/data/'
def load(name):
    p = f'../data/{name}'
    return pd.read_csv(p) if os.path.exists(p) else pd.read_csv(RAW + name)
print('準備OK')
from scipy import stats

# 発展マーケ-04. チャネル別クリック率の比較（2標本t検定）

> 🟢 Colab（ブラウザ）で実行できます。**最初に「① セットアップ」セルを実行**してください。
> 📌 **統計検定2級レベル**（実務でよく使う検定・回帰）。scipy・statsmodels を使います（Colabは導入済み。ローカルは `uv add scipy statsmodels`）。

**舞台設定**：「紹介」と「Web広告」では、クリック率(CTR)に**本当に**差があるのか。見た目の平均だけでなく、**その差が偶然では説明できないか**を2標本t検定で確かめます。

**使う統計（2級）**：2標本t検定（平均の差の検定）・p値・有意水準。

### 📋 使うデータ：`web_marketing.csv`（月×チャネルのマーケ実績30行）
1行＝ある月・あるチャネルの実績。`クリック数 / 表示回数 = CTR(クリック率)` を比べます。

| 月 | チャネル | 表示回数 | クリック数 | 獲得数 | 費用 |
|---|---|---|---|---|---|
| 2026-01 | 展示会 | 1855 | 149 | 22 | 4363 |

## 🎯 この章でできるようになること
- 2つのグループの平均の差を **2標本t検定** で検定できる
- p値と有意水準から「差は偶然か」を判断できる
- 分散が違う場合の **ウェルチのt検定** を使える

> **前提**：統計2級（仮説検定）　/　**所要**：約25分　/　**レベル**：2級

## 1. 各チャネルのCTR（クリック率）を出す

In [ ]:
mk = load('web_marketing.csv')
mk['CTR'] = mk['クリック数'] / mk['表示回数']            # クリック率＝クリック÷表示
print(mk.groupby('チャネル')['CTR'].agg(['mean','std','count']).round(4))

## 2. 「見た目の差」は偶然か？

紹介とWeb広告のCTR平均には差がありそうですが、月ごとのばらつきもあります。

- H₀（帰無仮説）：2チャネルの**母平均CTRは等しい**（差は偶然）
- H₁（対立仮説）：母平均CTRは異なる

分散が等しいとは限らないので、**ウェルチのt検定**（`equal_var=False`）を使います。

In [ ]:
a = mk[mk['チャネル']=='紹介']['CTR']
b = mk[mk['チャネル']=='Web広告']['CTR']
t, p = stats.ttest_ind(a, b, equal_var=False)            # ウェルチのt検定
print(f'紹介 CTR平均   = {a.mean():.3f}')
print(f'Web広告 CTR平均 = {b.mean():.3f}')
print(f'\nt統計量 = {t:.2f},  p値 = {p:.5f}')
print('→ 有意水準5%で', ('有意差あり（偶然では説明できない）' if p < 0.05 else '有意差なし'))

## 3. 差の大きさも見る（検定だけで終わらない）

「有意かどうか」と「どれくらい違うか」は別の話。差の大きさ（実務インパクト）も必ず確認します。

In [ ]:
diff = a.mean() - b.mean()
print(f'CTRの差 = {diff:.3f}（{diff/b.mean():.0%} の差）')
fig, ax = plt.subplots(figsize=(6,4))
ax.boxplot([a, b], tick_labels=['紹介','Web広告'])
ax.set_ylabel('CTR'); ax.set_title('チャネル別CTRの分布'); plt.show()

## 🧭 まとめ
- 2グループの平均の差は **2標本t検定** で「偶然か」を判定する。
- 分散が違いそうなら **ウェルチのt検定**（`equal_var=False`）が安全。
- **有意性（p値）** と **差の大きさ** は別物。両方を見て意思決定する。

> ⚠️ **よくある間違い**
> - p値は「効果の大きさ」ではない。n が大きいとごく小さな差でも有意になる。
> - 多くのチャネルを総当たりでt検定すると、偶然の有意（偽陽性）が増える → 3チャネル以上は**分散分析**(次章)。
> - 「有意差なし」は「差が無いと証明された」ではない（検出力不足の可能性）。

## 🧠 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます。

In [ ]:
# 採点用の関数（答え合わせに使うだけ・答えは表示しません）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
# Q1: [12,14,16] と [10,11,12] の平均の差（前−後）を ans に
import numpy as np
ans = None   # 例: np.mean([12,14,16]) - np.mean([10,11,12])
_check('Q1 平均の差', ans, np.mean([12,14,16]) - np.mean([10,11,12]))

In [ ]:
# Q2: p値が 0.02 のとき、有意水準5%で『有意』なら1・『有意でない』なら0 を ans に
ans = None   # ヒント: 0.02 < 0.05 ?
_check('Q2 有意判定', ans, 1)

In [ ]:
from scipy import stats
# Q3: [1,2,3,4] と [5,6,7,8] の2標本t検定のp値を ans に
ans = None   # 例: stats.ttest_ind([1,2,3,4],[5,6,7,8]).pvalue
_check('Q3 t検定のp値', ans, stats.ttest_ind([1,2,3,4],[5,6,7,8]).pvalue)

---
## 🏆 練習問題

**問1.** 「展示会」と「メルマガ」のCTRに差があるか、同じ手順でt検定しよう。

**問2.** CTRではなく **CVR（獲得数/クリック数）** で「紹介 vs Web広告」を検定しよう。結論は変わる？

**問3.**（考察）有意水準を5%から1%に厳しくすると、結論が変わるケースはどんなときか説明しよう。

> 🔑 **解答例は別ページ**（ネタバレ防止）👉 **[解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/04_%E3%83%81%E3%83%A3%E3%83%8D%E3%83%AB%E5%88%A5%E3%82%AF%E3%83%AA%E3%83%83%E3%82%AF%E7%8E%87%E3%81%AE%E6%AF%94%E8%BC%83.md)**
>
> 自分で解いてから開きましょう。

## 📒 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 帰無仮説 H₀ | 「差がない」とする仮定 |
| p値 | H₀が正しいとき今回以上の差が出る確率 |
| 有意水準 | 偶然と切り捨てる基準（通常5%） |
| ウェルチのt検定 | 分散が等しくなくても使えるt検定 |

```python
from scipy import stats
stats.ttest_ind(a, b, equal_var=False)   # 2標本t検定（ウェルチ）
```